# Session 9 — Machine learning foundations for EO
*Séance 9 — Fondements de l'apprentissage automatique pour l'OT*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/09_ml_fondements.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

| Training track · Parcours de formation | This session · Cette séance |
|---|---|
| Geomatics / environment · Géomatique / environnement (UdeS géomatique appliquée; Bishop's Environmental Studies & Geography) | ● full session / séance complète |
| Computer science · Informatique (UdeS informatique; Bishop's Computer Science) | ● full session / séance complète |
| College / continuing education · Cégep / formation continue | ● full session / séance complète |

**By the end of this session you will be able to:**

1. Frame a land-cover mapping problem as supervised classification (features, labels, samples).
2. Diagnose under- and over-fitting with training and test scores.
3. Compare classifiers fairly with cross-validation and pipelines.
4. Explain why random splits overestimate accuracy with spatially autocorrelated samples, and use grouped (spatial) cross-validation.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

## Part A — Lecture

**Supervised learning** learns a function *f(features) → label* from examples. In EO, a sample is usually a **pixel**, its
features are band reflectances and indices, and its label comes from field work or photo-interpretation. Analysts
digitize **training polygons** and take many pixels inside each one — so pixels of the same polygon look alike.
The dataset below simulates exactly this (60 polygons × 25 pixels, 5 classes).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ml4eo_data import training_polygons, BANDS, CLASSES

df = training_polygons()
FEATURES = BANDS + ["NDVI", "NDWI", "NDBI"]
X, y, groups = df[FEATURES], df["label"], df["polygon"]
print(df.shape)
df.groupby("label")[["B4", "B8", "B11", "NDVI"]].mean().round(3)

In [ ]:
ax = df.plot.scatter(x="x_km", y="y_km", c=df["label"].astype("category").cat.codes, cmap="tab10", s=4, figsize=(5, 5))
ax.set_title("Training pixels come in clusters (polygons)"); plt.show()

### Under- and over-fitting
A decision tree that grows without limit memorizes the training set. Compare training and test accuracy as depth grows.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
depths = range(1, 16)
train_acc = [DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr).score(X_tr, y_tr) for d in depths]
test_acc = [DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr).score(X_te, y_te) for d in depths]
plt.plot(depths, train_acc, "o-", label="training"); plt.plot(depths, test_acc, "s-", label="test")
plt.xlabel("Tree depth"); plt.ylabel("Accuracy"); plt.legend(); plt.title("Bias–variance trade-off"); plt.show()

### Fair comparison: pipelines and cross-validation
Scaling must be learned **inside** each training fold (a `Pipeline`), otherwise information leaks from the test fold.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold

models = {"k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(7)),
          "SVM (RBF)": make_pipeline(StandardScaler(), SVC(C=10, gamma="scale")),
          "Random forest": RandomForestClassifier(300, random_state=0, n_jobs=-1)}
random_cv = KFold(5, shuffle=True, random_state=0)
pd.DataFrame({name: cross_val_score(m, X, y, cv=random_cv) for name, m in models.items()}).describe().loc[["mean", "std"]].round(3)

## Part B — Lab

**Task 1 — Spatial cross-validation.** Pixels of the same polygon must never be split between training and test. Use
`GroupKFold(5)` with `groups` to compute the mean accuracy of each model into a dict `spatial_scores`, and the random
5-fold means into `random_scores`.

In [ ]:
# YOUR CODE HERE
spatial_scores = None
random_scores = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(set(spatial_scores) == set(models))
check(all(random_scores[k] > spatial_scores[k] for k in models), "Random CV should look optimistic for every model")

**Task 2 — Feature importance.** Fit the random forest on all data and compute **permutation importance** on a grouped
hold-out (`GroupShuffleSplit`, 30% of polygons). Store a sorted Series `importance`.

In [ ]:
# YOUR CODE HERE
importance = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(len(importance) == len(FEATURES))
check(importance.index[0] in FEATURES)

**Task 3 — Learning curve.** How many polygons do we need? Train the random forest on 10, 20, 30 and 40 polygons (first
polygons by id) and evaluate on polygons 50–59. Store accuracies in a dict `curve`.

In [ ]:
# YOUR CODE HERE
curve = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(sorted(curve) == [10, 20, 30, 40])
check(curve[40] >= curve[10] - 0.02, "More polygons should not make things clearly worse")

**❓ Question —** Your colleague reports 99% accuracy from a random 70/30 split of pixels. What would you ask before trusting it?

*Your answer here.*

## Summary

- Training accuracy is not a quality measure; use held-out data.
- Put preprocessing in pipelines so cross-validation is honest.
- With clustered samples, split by polygon (or by spatial block): random splits are optimistic.

**To go further:**

- Roberts, D. R. et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography*, 40, 913–929.
- Maxwell, Warner & Fang (2018), *IJRS* 39(9) — applied review of ML classification in remote sensing.